# Train MAiSTRO's transformer, and export it for serverless deployment

Run this on **Colab with a GPU** (`Runtime → Change runtime type → T4 GPU`). On a T4 an epoch takes a
couple of minutes; on your laptop's CPU it takes ~17.

What comes out the other end is two small files:

| File | Size | What it is |
|---|---|---|
| `transformer.npz` | ~7 MB | The trained weights, in float16 |
| `vocabulary.npz` | ~0.2 MB | The 3,388 note tokens, the corpus to seed from, its pitch histogram |

Those two files are everything the deployed API needs. It runs the forward pass in **NumPy** —
no TensorFlow — because TensorFlow is 877 MB unpacked and requires Python ≤ 3.10, while Vercel's
Python runtime is 3.12 with a 500 MB ceiling. The whole serving bundle comes to about 40 MB.

**Before you run this**, push the branch so Colab can clone it.

In [ ]:
#@title 1. Confirm we have a GPU
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "No GPU — switch runtime type!")

In [ ]:
#@title 2. Clone the repo and install what training needs
REPO   = "https://github.com/DasolLim/MAiSTRO-AI.git"  #@param {type:"string"}
BRANCH = "feat/sampling-conditioning-arena"            #@param {type:"string"}

![ -d MAiSTRO ] || git clone --depth 1 --branch $BRANCH $REPO MAiSTRO
%cd /content/MAiSTRO

# music21 is only needed to parse MIDI; data/notes is committed, so it is used from cache.
# keras-self-attention is NOT installed: architectures.py imports it lazily, and only the
# LSTM+attention model needs it. That is what lets this notebook run on Colab's Keras 3.
!pip install -q music21 mido

import tensorflow as tf, keras
print(f"tensorflow {tf.__version__} | keras {keras.__version__} | GPUs {tf.config.list_physical_devices('GPU')}")

In [ ]:
#@title 3. Look at the corpus
import pickle
from backend.maistro import config

notes = pickle.load(open(config.NOTES_FILE, "rb"))
pitchnames = sorted(set(notes))
n_vocab = len(pitchnames)

print(f"{len(notes):,} note/chord/rest tokens")
print(f"{n_vocab:,} unique tokens")
print(f"{len(notes) - config.SEQUENCE_LENGTH:,} training windows of length {config.SEQUENCE_LENGTH}")
print(f"\nfirst 8 tokens: {notes[:8]}")

In [ ]:
#@title 4. Train
#@markdown A tenth of the corpus is held out for validation. `val_loss` is the honest number:
#@markdown training loss keeps falling long after the model starts memorising 200 MIDI files.
EPOCHS     = 20   #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}

from backend.maistro.train import train_model

result = train_model(
    arch="transformer",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    checkpoint_every_n_epochs=2,
    on_epoch_end=lambda e, total, loss, val: print(
        f"  epoch {e:>3}/{total}  loss {loss:.4f}" + (f"  val_loss {val:.4f}" if val else "")
    ),
)
print(f"\ncheckpoints in {result.checkpoint_dir}")

In [ ]:
#@title 5. Plot the loss curves
#@markdown If `val_loss` turns upward while `loss` keeps falling, you have passed the useful point.
import matplotlib.pyplot as plt

history = result.history
plt.figure(figsize=(7, 4))
plt.plot(history["loss"], label="train")
if "val_loss" in history:
    plt.plot(history["val_loss"], label="validation")
    best = min(range(len(history["val_loss"])), key=lambda i: history["val_loss"][i])
    plt.axvline(best, ls=":", c="grey")
    plt.title(f"best val_loss {history['val_loss'][best]:.4f} at epoch {best + 1}")
plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.legend(); plt.grid(alpha=.3)
plt.show()

In [ ]:
#@title 6. Listen before you ship
#@markdown Generate a short piece with the sampling settings the app defaults to.
#@markdown Untrained output sounds like noise; a trained model should not.
from backend.maistro.generate import GenerationConfig, generate_tokens
from backend.maistro import metrics

tokens = generate_tokens(GenerationConfig(arch="transformer", n_notes=160, seed=1))
print(f"repetition rate : {metrics.repetition_rate(tokens):.3f}   (greedy LSTM baseline ≈ 0.151)")
print(f"corpus KL       : {metrics.evaluate(tokens)['pitch_class_kl']:.3f}   (lower = closer to Chopin)")
print(f"unique tokens   : {len(set(tokens)) / len(tokens):.3f}")
print(f"\nfirst 12: {tokens[:12]}")

In [ ]:
#@title 7. Export to NumPy, and prove it matches Keras
#@markdown The deployed API never runs TensorFlow, so the exported weights must reproduce
#@markdown the Keras model exactly. If this check fails, do not ship.
from pathlib import Path
import numpy as np
from backend.maistro import architectures, export, npmodel

OUT = Path("web/model")
model = architectures.load_trained_network("transformer", n_vocab)

weights_path = export.export_weights(model, OUT)          # float16
vocab_path   = export.export_vocabulary(OUT)

rng = np.random.default_rng(0)
probe = rng.integers(0, n_vocab, size=config.SEQUENCE_LENGTH).astype(np.int32)
keras_probs = np.asarray(model(probe[None, :], training=False))[0]
numpy_probs = npmodel.predict_next(npmodel.load(weights_path), probe)

drift = float(np.abs(keras_probs - numpy_probs).max())
agree = int(keras_probs.argmax()) == int(numpy_probs.argmax())
print(f"max |keras - numpy| : {drift:.2e}")
print(f"argmax agrees       : {agree}")
print(f"weights             : {weights_path.stat().st_size / 1e6:.2f} MB")
print(f"vocabulary          : {vocab_path.stat().st_size / 1e6:.2f} MB")

assert agree and drift < 1e-4, "NumPy export does not match Keras — do not deploy this."
print("\n✓ export verified")

In [ ]:
#@title 8. Download the two files
#@markdown Drop them into `web/model/` in your local checkout, then commit.
from google.colab import files
files.download(str(weights_path))
files.download(str(vocab_path))

## Next

1. Put `transformer.npz` and `vocabulary.npz` into `web/model/` locally.
2. Optionally keep the full Keras checkpoint too (`checkpoints/transformer/`) — it is what the
   **arena** needs to run a blind A/B against the LSTM. It is *not* needed to deploy.
3. Deploy: the serverless API reads only those two files.